# **Повторяем подготовку данных.**

In [77]:
#Импортируем библиотеки
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.style.use('ggplot')
%matplotlib inline
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

#Загружаем данные
data = './Econom_Cities_data.csv'
df = pd.read_csv(data, sep=';', decimal=',',  index_col='City')
df

,Work,Price,Salary
City,,,
Amsterdam,1714,65.6,49.0
Athens,1792,53.8,30.4
Bogota,2152,37.9,11.5
Bombay,2052,30.3,5.3
Brussels,1708,73.8,50.5
Buenos_Aires,1971,56.1,12.5
Cairo,-9999,37.1,-9999.0
Caracas,2041,61.0,10.9
Chicago,1924,73.9,61.9


Стандартизация данных и удаление отрицательных значений.

In [78]:
df = df.drop(['Cairo', 'Jakarta'])
X = df
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# **DB-scan**



In [79]:
# Создадим объект DBSCAN при eps = 2
dbscan = DBSCAN(eps=2, metric='euclidean', min_samples=2)
# Обучаем модель DBSCAN
dbscan.fit(X_scaled)

unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[ 0 46]]


Все объекты оказались в одном кластере. Такой вариант не подходит. Необходимо уменьшить eps.

In [80]:

# Создадим объект DBSCAN при eps = 1
dbscan = DBSCAN(eps=1, metric='euclidean', min_samples=2)
# Обучаем модель DBSCAN
dbscan.fit(X_scaled)

unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)


[[-1  4]
 [ 0 38]
 [ 1  2]
 [ 2  2]]


 В 0 кластере слишком много объектов по сравненению с другими. Пробуем дальше.

In [83]:

# Создадим объект DBSCAN при eps = 0.9
dbscan = DBSCAN(eps=0.8, metric='euclidean', min_samples=3)
# Обучаем модель DBSCAN
dbscan.fit(X_scaled)

unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[-1  8]
 [ 0 19]
 [ 1 19]]


Выходит слишком много шумов.

In [84]:
# Создадим объект DBSCAN при eps = 0.75 и min_samples=2
dbscan = DBSCAN(eps= 0.75, metric='euclidean', min_samples=2)
# Обучаем модель DBSCAN
dbscan.fit(X_scaled)

unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[-1  4]
 [ 0 19]
 [ 1 19]
 [ 2  2]
 [ 3  2]]


Более удачный результат.

In [87]:
# Создадим объект DBSCAN при eps = 0.75 и min_samples=2
dbscan = DBSCAN(eps= 0.6, metric='euclidean', min_samples=2)
# Обучаем модель DBSCAN
dbscan.fit(X_scaled)

unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[-1 10]
 [ 0 16]
 [ 1 14]
 [ 2  2]
 [ 3  2]
 [ 4  2]]


Теперь происходит уже излишнее разбиение на кластеры, которое тяжело будет интерпритировать, остановимся на eps = 0.75

In [88]:
# Создадим объект DBSCAN при eps = 0.75 и min_samples=2
dbscan = DBSCAN(eps= 0.75, metric='euclidean', min_samples=2)
# Обучаем модель DBSCAN
dbscan.fit(X_scaled)

unique, counts = np.unique(dbscan.labels_, return_counts=True)
print(np.asarray((unique, counts)).T)

[[-1  4]
 [ 0 19]
 [ 1 19]
 [ 2  2]
 [ 3  2]]


In [89]:
df['dbscan'] = dbscan.labels_
df['dbscan'].sort_values()

,dbscan
City,
Hong_Kong,-1
Tokyo,-1
Taipei,-1
Stockholm,-1
Dublin,0
Amsterdam,0
Chicago,0
Brussels,0
Houston,0


In [90]:

df.groupby('dbscan').mean()

,Work,Price,Salary
dbscan,,,
-1,2051.250000,93.600000,42.375000
0,1792.000000,77.526316,55.157895
1,1959.210526,50.115789,14.789474
2,1874.000000,97.950000,95.150000
3,1625.000000,114.550000,65.150000


В результате кластерного анализа с использованием алгоритма DBSCAN были идентифицированы четыре кластера и четыре точки, классифицированные как выбросы (шум).

 (-1) Сюда попали города, где фиксируется высокое рабочее время, высокие цены и зарплаты чуть ниже средних. Вероятно, местным жителям приходится много трудиться, чтобы поддерживать приемлемый уровень жизни.

 (0) Этот кластер объединяет города с наиболее гармоничной экономической структурой, где все ключевые показатели держатся на средних значениях.

 (1) Этот кластер характеризуется низким качеством жизни: здесь невысокие заработки (на фоне сравнительно дорогих товаров и услуг) и одновременно очень долгий рабочий день.

 (2) Этот кластер представляет города с высоким уровнем жизни- в них отмечаются высокие индексы цен и зарплат, при этом продолжительность рабочего дня остаётся умеренной.

 (3) Кластер 3 выделяется самым низким временем работы и самыми высокими ценами при средних зарплатах. Скорее всего, жители этих городов ценят свободное время больше, чем карьерные достижения.